## Robustness of networks using MILP and Fast-Lin

### Analyzes how the robustness of networks depends on variables such as dense vs CNN, number of layers, and width of layers.

Let $f: [0, 1]^{n_0} \to \mathbb{R}^{10}$ denote a neural network trained to classify numbers 0-9, where the network classifies image $x$ being labelled as $i$ more likely than $j$ if $f_i(x) > f_j(x)$.

Let the image $x_0$ be classified as $i$ by the network. We find non-trivial $\epsilon$ such that $f_i(x) > f_j(x)$ for all $x \in B_p(x_0, \epsilon)$ and $i \neq j$.

In [ ]:
import csv
from matplotlib import pyplot as plt
import pandas as pd
from pathlib import Path
import torch
from torchvision import datasets
from tqdm import tqdm

from src.architectures.networkArchitectures import networkRegistry
from src.robustness.exactRobustness import exactRobustness
from src.robustness.fastLin import fastLin
from src.training.networkTraining import transform
from src.utils.extractParams import extractParams
from src.utils.loadNetwork import loadNetwork

In [ ]:
testingSet = datasets.MNIST(root = "./data", train = False, transform = transform, download = True)
numberOfImages = 100    # Number of images to analyze for each network

Computed results are saved in the results folder. This allows us to use these without computing them every time.

In [ ]:
# Results are saved in a csv file. Columns are give by:
# "name", the name of the network
# "imageIndex", the index of the image
# "label", the true label of the image
# "predictedClass", the predicted label of the image
# "exactRobustness", whether robustness was computed by MILP or Fast-Lin
# "epsilon", the exact epsilon for exactRobustness and the certified lower bound otherwise

resultsDirectory = Path("results")
fileName = "robustnessResults.csv"
filePath = resultsDirectory / fileName

# Load already completed results
if filePath.exists():
    df = pd.read_csv(filePath)
    computedResults = set(zip(df["name"], df["imageIndex"], df["exactRobustness"]))
else:
    computedResults = set()

### Computes certified lower bounds

Gets certified lower bounds by Fast-Lin. Could be fast enough for all considered networks even unoptimized.

In [ ]:
# Open CSV for appending
fileExists = filePath.exists()

with filePath.open(mode = "a", newline = "") as file:
    writer = csv.writer(file)

    # Create header if file does not already exist
    if not fileExists:
        writer.writerow(["name", "imageIndex", "label", "predictedClass", "exactRobustness", "epsilon"])

    for networkName in networkRegistry:
        print(f"Computing Fast-Lin for network: {networkName}")
        network = loadNetwork(networkName)

        parameters = extractParams(network = network, inputShape = (1, 28, 28))
        weights = [W for W, _ in parameters]
        biases = [b for _, b in parameters]

        for imageIndex in tqdm(range(numberOfImages)):
            # Skip image if already computed for the network using Fast-Lin
            if (networkName, imageIndex, False) in computedResults:
                continue

            # Compute results
            image, label = testingSet[imageIndex]

            pNorm = 1
            x0 = image.view(-1).numpy()

            output = network(image)
            _, predictedClass = torch.max(output, 1)
            predictedClass = predictedClass.item()
            targetClasses = [targetClass for targetClass in range(0, 10) if targetClass != predictedClass]
            
            certifiedEpsilon, _, _ = fastLin(weights = weights, biases = biases, x0 = x0, pNorm = pNorm, epsilon0 = 10, originalClass = predictedClass, targetClasses = targetClasses, tolerance = 0.005)

            # Save computed results
            writer.writerow([networkName, imageIndex, label, predictedClass, False, certifiedEpsilon])
            file.flush()

Plot figures comparing robustness vs layer width, number of layers, and total number of neurons.

In [ ]:
# Plot robustness obtained by Fast-Lin

### Computes exact robustness

Get exact robustness by solving multiple MILPs created via the big-M formulation. Infeasible for larger networks.

In [ ]:
# Open CSV for appending
fileExists = filePath.exists()

with filePath.open(mode = "a", newline = "") as file:
    writer = csv.writer(file)

    # Create header if file does not already exist
    if not fileExists:
        writer.writerow(["name", "imageIndex", "label", "predictedClass", "exactRobustness", "epsilon"])

    for networkName in networkRegistry:
        print(f"Computing MILP for network: {networkName}")
        network = loadNetwork(networkName)

        parameters = extractParams(network = network, inputShape = (1, 28, 28))
        weights = [W for W, _ in parameters]
        biases = [b for _, b in parameters]

        for imageIndex in tqdm(range(numberOfImages)):
            # Skip image if already computed for the network using Fast-Lin
            if (networkName, imageIndex, True) in computedResults:
                continue

            # Compute results
            image, label = testingSet[imageIndex]

            pNorm = 1
            x0 = image.view(-1).numpy()

            output = network(image)
            _, predictedClass = torch.max(output, 1)
            predictedClass = predictedClass.item()
            targetClasses = [targetClass for targetClass in range(0, 10) if targetClass != predictedClass]
            
            epsilon, _, _ = exactRobustness(weights = weights, biases = biases, x0 = x0, pNorm = pNorm, epsilon0 = 10, originalClass = predictedClass, targetClasses = targetClasses)

            # Save computed results
            writer.writerow([networkName, imageIndex, label, predictedClass, True, epsilon])
            file.flush()

Plot figures comparing robustness vs layer width, number of layers, and total number of neurons for some smaller networks.

In [ ]:
# Plot robustness obtained by MILP